# 19 — Wholesale source-availability audit

**Question:** do the existing caches contain enough compatible wholesale history to justify a later feature experiment?

Select the verified Python 3.12 project kernel, then **Restart Kernel and Run All**. This optional audit uses the integrated source and, when available, one compatible complete 2024 cache. No downloads, model fitting, labels, prepared test files or dashboard changes occur. Only **2024–2025 rows** are loaded from source Parquet files. Missing 2024 history is reported rather than downloaded.

This is a source-coverage diagnostic, not a model experiment. Retail reference weeks are valid source observations, **not the model-ready eligibility population**. The audit retains the project's normalised item units and exact `Borong` category. Other labels are displayed for review, not automatically reclassified. Wholesale and retail prices may have different commercial meanings even when their recorded units match.

In [ ]:
import sys
from pathlib import Path
repository = Path.cwd().resolve()
if not (repository / "pricecatcher_config.py").is_file():
    repository = repository.parent
if not (repository / "pricecatcher_config.py").is_file():
    raise FileNotFoundError("Open this notebook inside the FYP repository.")
if str(repository) not in sys.path:
    sys.path.insert(0, str(repository))

import hashlib
import json
from importlib import reload
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import pricecatcher_config as cfg
cfg = reload(cfg)

START, END = "2024-01-01", "2025-12-31"
KEYS = ["project_group", "item_code", "unit", "state"]
WEEK_KEYS = KEYS + ["week_start"]
COLUMNS = ["date", "project_group", "item_code", "unit", "state", "premise_type", "premise_code", "price"]


def file_hash(path):
    with Path(path).open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()


cfg.verify_source_snapshot()
sources = [("integrated", cfg.STAGE1_COMBINED_PARQUET)]
protected = {cfg.STAGE1_DIR / "source_manifest.json": file_hash(cfg.STAGE1_DIR / "source_manifest.json")}
lookup_hashes = {name: file_hash(cfg.LOOKUP_DIR / name)
                 for name in ["lookup_item.parquet", "lookup_premise.parquet"]}
protected.update({cfg.LOOKUP_DIR / name: value for name, value in lookup_hashes.items()})
compatible = []
for path in sorted((cfg.DATA_HOME / "12_longer_history" / "2024_source").glob("*/source_manifest.json")):
    record = json.loads(path.read_text())
    policy = record["policy"]
    if (policy.get("lookup_sha256") == lookup_hashes
            and policy.get("item_map") == {str(k): v for k, v in cfg.PROJECT_ITEM_MAP.items()}
            and policy.get("months") == pd.period_range("2024-01", "2024-12", freq="M").astype(str).tolist()):
        source = path.parent / "selected_prices_2024.parquet"
        if not source.is_file() or file_hash(source) != record["selected_sha256"]:
            raise ValueError("Matching 2024 cache is missing or changed. Restore its original files.")
        compatible.append((path, source))
if len(compatible) > 1:
    raise ValueError("Multiple compatible 2024 caches exist. Review their provenance before selecting one.")
if compatible:
    manifest, source = compatible[0]
    protected[manifest] = file_hash(manifest)
    sources.append(("historical_2024", source))
else:
    print("No compatible complete 2024 cache: audit will report integrated-source coverage only.")
for _, source in sources:
    protected[source] = file_hash(source)
print("Run:", cfg.PROJECT_ROOT)
print("Sources:", [str(path) for _, path in sources])

## 1. Read training-period source records

Read the date-filtered columns through Arrow. For each source, report invalid prices and missing metadata before excluding them from this diagnostic. Dates outside 2024–2025 are not converted into a pandas table. Only finite positive prices, known basket codes and nonblank identifiers enter the weekly summaries. No statistical outlier removal or retrospective manual corrections are applied.

In [ ]:
def read_training_source(path):
    dataset = ds.dataset(path, format="parquet")
    date_type = dataset.schema.field("date").type
    if pa.types.is_timestamp(date_type) or pa.types.is_date(date_type):
        lower = pa.scalar(pd.Timestamp(START).to_pydatetime(), type=date_type)
        upper = pa.scalar(pd.Timestamp("2026-01-01").to_pydatetime(), type=date_type)
    else:
        raise ValueError("Expected typed calendar dates in the normalised source cache.")
    condition = (ds.field("date") >= lower) & (ds.field("date") < upper)
    return dataset.to_table(columns=COLUMNS, filter=condition).to_pandas()


frames, source_rows, premise_rows = [], [], []
for label, path in sources:
    data = read_training_source(path)
    data["date"] = pd.to_datetime(data.date, errors="raise").dt.normalize()
    if label == "historical_2024" and not data.date.dt.year.eq(2024).all():
        raise ValueError("The historical cache must contain only 2024 records.")
    for column in ["project_group", "unit", "state", "premise_type"]:
        data[column] = data[column].astype("string").str.strip().replace("", pd.NA)
    prices = pd.to_numeric(data.price, errors="coerce")
    invalid = prices.isna() | ~np.isfinite(prices) | (prices <= 0)
    missing = data[COLUMNS].drop(columns=["price"]).isna().any(axis=1)
    known = data.item_code.isin(cfg.SCOPE_ITEM_CODES)
    premise_rows.append(data.groupby("premise_type", dropna=False).size().rename("rows").reset_index().assign(source=label))
    valid = data.loc[~invalid & ~missing & known].copy()
    valid["price"] = prices.loc[valid.index]
    valid["source"] = label
    source_rows.append({"source": label, "training_period_rows": len(data),
        "invalid_price_rows": int(invalid.sum()), "missing_metadata_rows": int(missing.sum()),
        "outside_basket_rows": int((~known).sum()), "usable_rows": len(valid),
        "borong_rows": int(valid.premise_type.eq("Borong").sum()),
        "first_date": str(valid.date.min()), "last_date": str(valid.date.max()),
        "observed_months": int(valid.date.dt.to_period("M").nunique())})
    frames.append(valid)
source_summary = pd.DataFrame(source_rows)
premise_types = pd.concat(premise_rows, ignore_index=True)
daily = pd.concat(frames, ignore_index=True)
if daily.empty:
    raise ValueError("No usable training-period source records.")
if daily.duplicated(["date", "premise_code", "item_code"]).any():
    raise ValueError("Duplicate source keys: investigate overlap before combining caches.")
if not daily.project_group.eq(daily.item_code.map(cfg.PROJECT_ITEM_MAP)).all():
    raise ValueError("Source project-group labels disagree with the configured item map.")
daily["week_start"] = daily.date - pd.to_timedelta(daily.date.dt.dayofweek, unit="D")
# Exclude partial boundary weeks; this does not assert that every day was observed.
period_start = pd.Timestamp(START if compatible else "2025-01-01")
daily = daily.loc[(daily.week_start >= period_start)
                  & (daily.week_start + pd.Timedelta(days=6) <= pd.Timestamp(END))].copy()
display(source_summary)
display(premise_types)

## 2. Summarise wholesale coverage and lagged availability

Weekly wholesale prices give each observed shop equal weight: median within each shop/week, then median across shops for the same item/unit/state. Shop counts of at least two are an **audit comparison**, not a proven quality threshold.

For a retail origin week *t*, a candidate lagged wholesale change uses exactly *t−1* and *t−2*. Gaps remain missing; no forward filling or nearest-week matching occurs. Requiring two shops means two in **both** wholesale weeks, but does not guarantee the same shops. No lead–lag predictive claim is tested here.

In [ ]:
wholesale = daily.loc[daily.premise_type.eq("Borong")].copy()
shops = wholesale.groupby(WEEK_KEYS + ["premise_code"], observed=True).price.median().rename("shop_median").reset_index()
weekly = shops.groupby(WEEK_KEYS, observed=True).agg(
    wholesale_price=("shop_median", "median"), shops=("premise_code", "nunique")).reset_index()
previous = weekly[WEEK_KEYS + ["wholesale_price", "shops"]].copy()
previous["week_start"] += pd.Timedelta(days=7)
previous = previous.rename(columns={"wholesale_price": "previous_price", "shops": "previous_shops"})
weekly = weekly.merge(previous, on=WEEK_KEYS, how="left", validate="one_to_one")
weekly["wholesale_change_pct"] = (weekly.wholesale_price / weekly.previous_price - 1) * 100
weekly["two_shop_change"] = weekly.shops.ge(2) & weekly.previous_shops.ge(2)
weekly["year"] = weekly.week_start.dt.year
wholesale_support = weekly.groupby(KEYS + ["year"], observed=True).agg(
    observed_weeks=("week_start", "nunique"), weeks_two_shops=("shops", lambda x: int(x.ge(2).sum())),
    consecutive_changes=("wholesale_change_pct", "count"), median_shops=("shops", "median")).reset_index()

retail_weeks = daily.loc[~daily.premise_type.eq("Borong"), WEEK_KEYS].drop_duplicates()
lagged = weekly[WEEK_KEYS + ["wholesale_price", "wholesale_change_pct", "two_shop_change"]].copy()
lagged["week_start"] += pd.Timedelta(days=7)
matched = retail_weeks.merge(lagged, on=WEEK_KEYS, how="left", validate="one_to_one")
matched["year"] = matched.week_start.dt.year
matched["has_lagged_price"] = matched.wholesale_price.notna()
matched["has_lagged_change"] = matched.wholesale_change_pct.notna()
matched["has_two_shop_change"] = matched.two_shop_change.eq(True)
coverage = matched.groupby(KEYS + ["year"], observed=True).agg(
    retail_source_weeks=("week_start", "size"), lagged_price_weeks=("has_lagged_price", "sum"),
    lagged_change_weeks=("has_lagged_change", "sum"), two_shop_change_weeks=("has_two_shop_change", "sum")).reset_index()
for name in ["lagged_price", "lagged_change", "two_shop_change"]:
    coverage[name + "_coverage_pct"] = coverage[name + "_weeks"] / coverage.retail_source_weeks * 100
summary = coverage.groupby(["project_group", "year"], observed=True)[
    ["retail_source_weeks", "lagged_price_weeks", "lagged_change_weeks", "two_shop_change_weeks"]].sum().reset_index()
for name in ["lagged_price", "lagged_change", "two_shop_change"]:
    summary[name + "_coverage_pct"] = summary[name + "_weeks"] / summary.retail_source_weeks * 100
print("Usable Borong rows:", len(wholesale), "| Wholesale state/item/unit weeks:", len(weekly))
display(summary)
display(wholesale_support.head(20))

## 3. Save the compact audit

Repeat runs with identical inputs reuse verified outputs. A different source snapshot creates a separate audit folder. The source caches, models and deployment pointers are never edited.

In [ ]:
policy = {"schema_version": 1, "period": [START, END], "wholesale_category": "Borong",
    "item_map": {str(k): v for k, v in cfg.PROJECT_ITEM_MAP.items()},
    "group_keys": KEYS, "price": "equal-shop weekly median", "change": "exact consecutive weeks",
    "lag_weeks": 1, "comparison_minimum_shops": 2,
    "population": "source retail item/unit/state weeks, not model-ready rows",
    "historical_cache_available": bool(compatible), "publication_delays_modelled": False}
audit_notebook = json.loads((repository / "research" / "19_wholesale_availability_audit.ipynb").read_text())
policy["code_sha256"] = hashlib.sha256("\n".join(
    "".join(cell["source"]) for cell in audit_notebook["cells"] if cell["cell_type"] == "code").encode()).hexdigest()
inputs = {cfg.input_reference(path): value for path, value in protected.items()}
identity = hashlib.sha256(json.dumps({"policy": policy, "inputs": inputs}, sort_keys=True).encode()).hexdigest()[:16]
output = cfg.PROJECT_ROOT / "research" / "19_wholesale_availability_audit" / identity
for path, expected in protected.items():
    if file_hash(path) != expected:
        raise ValueError("Source input changed while auditing. No outputs committed.")
tables = {"source_summary.csv": source_summary, "premise_types.csv": premise_types,
          "wholesale_support.csv": wholesale_support, "retail_signal_coverage.csv": coverage,
          "group_signal_coverage.csv": summary}
manifest_path = output / "audit_manifest.json"
if manifest_path.exists():
    saved = json.loads(manifest_path.read_text())
    if saved["policy"] != policy or saved["input_sha256"] != inputs or set(saved["output_sha256"]) != set(tables):
        raise ValueError("Existing audit describes different settings or inputs.")
    for name, expected in saved["output_sha256"].items():
        if not (output / name).is_file() or file_hash(output / name) != expected:
            raise ValueError("Saved audit output missing or changed. Preserve it for investigation.")
    print("Reusing verified audit outputs.")
else:
    output.mkdir(parents=True, exist_ok=True)
    for name, table in tables.items():
        table.to_csv(output / name, index=False)
    manifest_path.write_text(json.dumps({"policy": policy, "input_sha256": inputs,
        "output_sha256": {name: file_hash(output / name) for name in tables}}, indent=2))
print("Saved:", output)
print("Share source_summary.csv and group_signal_coverage.csv first.")

## Interpretation and next decision

- Zero or sparse Borong coverage argues against prioritising a wholesale feature experiment with these caches. Do not guess missing wholesale prices or broaden geography solely to obtain higher coverage.
- Substantial overlap supports a later controlled feature test; it does **not** establish an early signal, reliable prices or improved AP. Record publication timing and changing-shop limitations.
- Review `retail_signal_coverage.csv` for uneven state/item coverage, rather than relying only on national totals. This reference population differs from model-ready rows; model-fold feature coverage would still need checking.
- Festival dates can be tested separately if justified, but no dates are invented or features added by this audit. Keep the existing label, folds and dashboard unchanged.
- Some cached historical metadata is provisional. Suspicious retained prices are not independently verified. A count threshold cannot resolve those uncertainties.